In [39]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [40]:
yerevan_historical = pd.read_csv('../data/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../data/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [46]:
yerevan_historical.drop(columns=[ 'days_in'], inplace=True)

In [47]:
yerevan_actual['backup_status'] = 'not sold'

In [80]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [81]:
df.drop(columns=['links', 'source', 'rent_or_sale'], inplace=True)

In [82]:
df[df['id']==10895]

,backup_status,id,price,rooms,square_meters,address,sold_date,furniture,renovation,price_per_meter,...,height,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered
33,sold,10895,100000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",2022-10-08 21:56:34,1.0,NaN,1851.851852,...,2.8,1.0,2022-08-28 19:49:22,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513121,40.166179
18279,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",NaN,1.0,NaN,2314.814815,...,2.8,1.0,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513120,40.166179


In [83]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [84]:
df = df.dropna(subset=['district']).reset_index(drop=True)

### classifying neighborhoods

In [55]:
df2 = df.head(80589).reset_index(drop=True)

In [32]:
import requests

api_key = "***REMOVED-GOOGLE-API-KEY***"

addresses = []

In [33]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            # print(address)
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error

In [87]:
df['neighborhood'] = addresses

In [63]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)

### small amount of rows that have yerevan classfied replaced with district value 

In [64]:
df = pd.read_csv('../data/yerevan_combined_clean/sale_combined.csv')

In [150]:
for index, row in df[df['neighborhood'] == 'Yerevan'].iterrows():
    df.loc[index, 'neighborhood'] = row['district']

In [151]:
df.loc[df['neighborhood'] == '0054', 'neighborhood'] = 'Davtashen Block'
df.loc[df['neighborhood'] == '0059', 'neighborhood'] = 'Nor Nork 9th Microdistrict'

df['neighborhood'] = df['neighborhood'].str.replace(' ', '_')
df['neighborhood'] = df['neighborhood'].str.replace('-', '_')

In [152]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [153]:
# removing neighborhoods with low counts
neighborhood_counts = sale_historical['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_historical = sale_historical[~sale_historical['neighborhood'].isin(neighborhoods_to_remove)]

In [154]:
# removing neighborhoods with low counts
neighborhood_counts = sale_actual['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_actual = sale_actual[~sale_actual['neighborhood'].isin(neighborhoods_to_remove)]

In [155]:
df = pd.concat([sale_historical, sale_actual], axis=0)

for some reason when scraping the data there were missing values for square-meters when price-per-meter had a value and the other way around

In [166]:
for index, row in df.iterrows():
    if np.isnan(row['price_per_meter']):
        continue
    else:
        df.loc[index, 'square_meters'] = row['price'] / row['price_per_meter']

In [167]:
for index, row in df.iterrows():
    if np.isnan(row['square_meters']):
        continue
    else:
        df.loc[index, 'price_per_meter'] = row['price'] / row['square_meters']

In [171]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)

In [14]:
df = pd.read_csv('../data/yerevan_combined_clean/sale_combined.csv')

In [67]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [73]:
filtered_df = pd.read_csv('../data/yerevan_historical_clean/sale_historical.csv')

In [86]:
df[df['backup_status']=='sold'].sort_values(by='square_meters',ascending=False)

,backup_status,id,price,rooms,square_meters,address,sold_date,furniture,renovation,price_per_meter,...,height,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered
5614,sold,10057211,320000.0,NaN,632.0,Yerevan › Nork-Marash,2022-12-06 17:20:56,0.0,Cosmetic Renovation,506.329114,...,NaN,3.0,2022-08-29 10:33:18,44.539855,40.179985,Yeritasardakan,4008,Norq_Marash,44.539858,40.179979
31319,sold,17734446,250000.0,4.0,630.0,Yerevan › Nork-Marash,2022-12-06 17:20:56,0.0,Major Renovation,396.825397,...,NaN,2.0,2022-09-13 04:31:56,44.539855,40.179985,Yeritasardakan,4008,Norq_Marash,44.539860,40.179994
11348,sold,13797535,95000.0,4.0,627.0,"Qanaqer 2nd Street, Yerevan",2022-08-08 20:25:49,NaN,NaN,NaN,...,NaN,1.0,2022-09-28 20:57:28,44.550626,40.187757,Yeritasardakan,4199,Norq_Marash,44.550617,40.187759
46410,sold,18319109,400000.0,NaN,625.0,Yerevan › Nork-Marash,2022-09-25 17:04:49,0.0,No Renovation,640.000000,...,NaN,3.0,2022-09-17 06:44:06,44.539855,40.179985,Yeritasardakan,4008,Norq_Marash,44.539846,40.179991
9501,sold,13217840,430000.0,NaN,620.0,"Armenak Armenakyan Street, Yerevan",2023-03-09 15:21:29,0.0,No Renovation,693.548387,...,NaN,2.0,2022-08-29 10:17:25,44.536307,40.186262,Yeritasardakan,2771,Norq_Marash,44.536310,40.186260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61041,sold,19119793,100000.0,5.0,NaN,"Մուշ թաղամաս, Երևան",2023-03-14 15:26:36,0.0,Euro Renovation,617.283951,...,NaN,2.0,2023-03-14 12:19:54,44.468161,40.181239,Barekamutyun,4232,Malatia_Sebastia,44.468157,40.181244
61042,sold,19120368,88000.0,6.0,NaN,"Արագածի փողոց, Երևան",2023-03-14 15:26:36,0.0,Partial Renovation,275.000000,...,NaN,2.0,2023-03-14 12:19:47,44.484258,40.159855,Garegin Nzhdeh Square,1366,Shengavit,44.484265,40.159856
61043,sold,19120399,88000.0,6.0,NaN,"Արագածի փողոց, Երևան",2023-03-14 15:26:36,0.0,Partial Renovation,273.291925,...,NaN,2.0,2023-03-14 12:19:45,44.484258,40.159855,Garegin Nzhdeh Square,1366,Shengavit,44.484261,40.159848
61044,sold,19121390,90000.0,4.0,NaN,"Նորագավիթ 13-րդ փողոց, Երևան",2023-03-14 15:26:36,0.0,No Renovation,204.545455,...,NaN,1.0,2023-03-14 11:50:11,44.466885,40.113095,Charbakh,5101,Shengavit,44.466877,40.113086


In [ ]:
df['neighborhood'].value_counts()

In [92]:
df6 = pd.read_csv('../data/yerevan_combined_clean/sale_combined.csv')

In [94]:
df6[df6['backup_status']=='sold']

,backup_status,id,price,rooms,square_meters,address,furniture,renovation,price_per_meter,floor,...,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood
0,sold,1055,130000.000000,2.0,50.0,"Yerevan, Center, Vardanants St",1.0,NaN,2600.000000,3.0,...,1.0,2022-09-10 02:04:46,44.521089,40.174314,Republic Square,921,Kentron,44.521080,40.174321,Kentron
1,sold,1991,95000.000000,2.0,90.0,"Yerevan, Arabkir, V.Papazyan St",NaN,NaN,1055.555556,4.0,...,1.0,NaN,44.502252,40.204200,Barekamutyun,1196,Arabkir,44.502243,40.204194,Nor_Arabkir
2,sold,4351,80000.000000,2.0,20.0,"Yerevan, Arabkir,",NaN,NaN,4000.000000,3.0,...,1.0,NaN,44.509420,40.207286,Barekamutyun,1903,Arabkir,44.509425,40.207294,Nor_Arabkir
3,sold,4420,110000.000000,3.0,42.0,"Yerevan, Arabkir, Aghbyur Serob St",NaN,NaN,2619.047619,NaN,...,1.0,NaN,44.503042,40.196823,Baghramyan,694,Arabkir,44.503051,40.196817,Nor_Arabkir
4,sold,4862,65000.000000,2.0,50.0,"Yerevan, Davtashen, Davtashen 1 district",NaN,NaN,1300.000000,11.0,...,1.0,NaN,44.488840,40.220763,Barekamutyun,3271,Davtashen,44.488842,40.220762,1st_Block
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58594,sold,19114415,102000.000000,3.0,87.0,"Davidashen 1-st block, Yerevan",0.0,Major Renovation,1172.413793,2.0,...,1.0,NaN,44.488840,40.220763,Barekamutyun,3271,Davtashen,44.488846,40.220766,1st_Block
58595,sold,19114417,310000.000000,3.0,127.0,"Rostom street, Yerevan",0.0,No Renovation,2440.944882,6.0,...,2.0,NaN,44.519158,40.164487,Zoravar Andranik,1273,Kentron,44.519149,40.164492,Kentron
58596,sold,19114432,84851.509858,3.0,80.0,"Նորաշեն թաղամաս, Երևան",0.0,Euro Renovation,1060.643873,2.0,...,1.0,NaN,44.468700,40.213498,Barekamutyun,4846,Achapnyak,44.468697,40.213488,Norashen
58597,sold,19114447,62000.000000,2.0,45.0,"Khaghag Don street, Yerevan",0.0,Major Renovation,1377.777778,7.0,...,1.0,NaN,44.512555,40.177628,Republic Square,202,Kentron,44.512563,40.177631,Kentron


In [17]:
sale_actual.drop(columns=[ 'sold_date'], inplace=True)

In [19]:
sale_historical = sale_historical.dropna(subset=['sold_date'])

In [24]:
sale_historical['sold_date'] = pd.to_datetime(sale_historical['sold_date'])

# Extract the date part from the datetime column
sale_historical['sold_date'] = sale_historical['sold_date'].dt.date

In [ ]:
sale_historical['sold_date'].value_counts().sort_index()

In [30]:
sale_historical.to_csv(r'../data/yerevan_combined_clean/historical_sold.csv', index=False)
sale_actual.to_csv(r'../data/yerevan_combined_clean/actual_not_sold.csv', index=False)

In [31]:
df = pd.concat([sale_historical, sale_actual], axis=0)

In [35]:
df.drop(columns=[ 'sold_date'], inplace=True)

In [36]:
df_clean = df.dropna().reset_index(drop=True)

In [38]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)
df_clean.to_csv(r'../data/yerevan_combined_clean/sale_clean_combined.csv', index=False)